# Pokemon battle prediction: final submissions

This notebook clones our repository (https://github.com/baharasadi/FDS-CH1-Pokemon), trains the three selected models on the whole training set and writes their submission files.

## Setup

Clone the code and install the package versions used for the submissions. Internet must be on.

In [ ]:
!rm -rf /tmp/FDS-CH1-Pokemon
!git clone https://github.com/baharasadi/FDS-CH1-Pokemon.git /tmp/FDS-CH1-Pokemon
!git -C /tmp/FDS-CH1-Pokemon log -1 --oneline
!pip install -q -r /tmp/FDS-CH1-Pokemon/requirements.txt

In [ ]:
import hashlib
import sys
from pathlib import Path

import numpy as np
import sklearn

sys.path.insert(0, "/tmp/FDS-CH1-Pokemon")

from make_submission import EXTRA_GROUPS, FINAL_MODELS, GROUPS
from src.data import load_all
from src.features import build_features, build_species_table, build_vocab, select_columns
from src.models import MODELS

OUT_DIR = Path("/kaggle/working")
print("scikit-learn", sklearn.__version__, "numpy", np.__version__)

## Features

Load the battles and build all the features used by the three models.

In [ ]:
train, test, chart, sample = load_all()
y = np.array([int(b["player_won"]) for b in train])
species = build_species_table(train + test)
vocab = build_vocab(train)
X_train = build_features(train, species, chart, vocab, GROUPS + EXTRA_GROUPS)
X_test = build_features(test, species, chart, vocab, GROUPS + EXTRA_GROUPS)
print(X_train.shape, X_test.shape)

In [ ]:
def write_submission(name):
    model_name, groups = FINAL_MODELS[name]
    columns = select_columns(X_train, groups)
    model = MODELS[model_name]()
    model.fit(X_train[columns], y)
    predictions = dict(zip([b["battle_id"] for b in test], model.predict(X_test[columns])))
    submission = sample.copy()
    submission["player_won"] = submission["battle_id"].map(predictions).astype(int)
    path = OUT_DIR / f"submission_{name}.csv"
    submission.to_csv(path, index=False)
    print(path, "features:", len(columns), "rows:", len(submission))

## Submission 1: ensemble v2 (public score 0.69400)

Soft voting of logistic regression, gradient boosting and extra trees on the first and second round features (459). This is our best model in cross-validation.

In [ ]:
write_submission("ensemble_v2")

## Submission 2: logistic regression (public score 0.68800)

Logistic regression on the first round features (421). This is a simple linear model, so it differs from the two tree-based submissions.

In [ ]:
write_submission("logreg")

## Submission 3: gradient boosting v2 (public score 0.69133)

Histogram gradient boosting on the first and second round features (459). This is the best single model in cross-validation.

In [ ]:
write_submission("hgb_v2")

## Check

Fingerprints of the files, to compare with the ones uploaded to the competition.

In [ ]:
for name in ("ensemble_v2", "logreg", "hgb_v2"):
    path = OUT_DIR / f"submission_{name}.csv"
    print(name, hashlib.sha256(path.read_bytes()).hexdigest())